In [2]:
import pandas as pd
import platform
import glob
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

In [3]:
# Check the operating system - file location is different if its Windows or OS
if platform.system() == 'Windows':
    path = "G:/My Drive/EarthEngineData"
else:
    # For Holden's Mac
    path = "/Users/holden/Personal Projects/flame-flame-fruit/FireData"

files = glob.glob(path + "/*.csv")

In [4]:
# Throw all the files into one large pandas dataframe (this took me 22m to run btw)
df_list = []
for file in files:
    # Best practice is probably to put this all in a try catch but it worked for me for now...
    df = pd.read_csv(file)
    
    # Only add some of the days with no fire, since with too many it will skew predictions (since fire is rare)
    no_fires = df[df['T21_max'] == 0].sample(frac=0.0013413685916579098, random_state=42) #choosing 10% - we can change this
    # Every day with fire
    fires = df[df['T21_max'] > 0]
    
    both = pd.concat([fires, no_fires])
    df_list.append(both)

#Combine all the dataframes into one
final_df = pd.concat(df_list, ignore_index=True) #ignore_index to reset the row numbers on each list
print("final dataset size: ", final_df.shape)

final dataset size:  (166105, 30)


In [5]:
# Parse grid cell x,y indices out of system:index (format: YYYYMMDD_x,y)
# These represent which 4x4km grid cell in Colorado the row belongs to
final_df[['grid_x', 'grid_y']] = final_df['system:index'].str.split('_').str[1].str.split(',', expand=True).astype(int)

# Extracts the month number from the date (1-12) and adds it as a new column
final_df['month'] = pd.to_datetime(final_df['date']).dt.month

# Extracts year from the date and adds it as a new column
# Not necessary but could be useful to see if fire danger is increasing over time as a side project
final_df['year'] = pd.to_datetime(final_df['date']).dt.year

# Creates a 0/1 column in case there is a fire
final_df['fire'] = (final_df['T21_max'] > 0).astype(int)


In [6]:
# This just helps visualize the data frame's structure
print(final_df.columns.tolist())
final_df.head(5)

['system:index', 'T21_max', 'T21_mean', 'T21_stdDev', 'aspect_max', 'aspect_mean', 'aspect_stdDev', 'date', 'elevation_max', 'elevation_mean', 'elevation_stdDev', 'erc_max', 'erc_mean', 'erc_stdDev', 'pr_max', 'pr_mean', 'pr_stdDev', 'rmin_max', 'rmin_mean', 'rmin_stdDev', 'slope_max', 'slope_mean', 'slope_stdDev', 'tmmx_max', 'tmmx_mean', 'tmmx_stdDev', 'vs_max', 'vs_mean', 'vs_stdDev', '.geo', 'grid_x', 'grid_y', 'month', 'year', 'fire']


,system:index,T21_max,T21_mean,T21_stdDev,aspect_max,aspect_mean,aspect_stdDev,date,elevation_max,elevation_mean,...,tmmx_stdDev,vs_max,vs_mean,vs_stdDev,.geo,grid_x,grid_y,month,year,fire
0,"20090106_119,1109",323.500000,217.290663,120.561332,109.0,103.590361,8.552128,2009-01-06,1719,1634.042169,...,0.890324,11.414818,9.866028,0.766249,"{""type"":""MultiPoint"",""coordinates"":[]}",119,1109,1,2009,1
1,"20090106_119,1110",323.500000,31.087087,120.561332,111.0,97.243243,9.154537,2009-01-06,1751,1667.360360,...,1.030267,10.506145,9.062752,0.698769,"{""type"":""MultiPoint"",""coordinates"":[]}",119,1110,1,2009,1
2,"20090112_137,1057",317.700012,31.492129,118.399804,46.0,26.851312,9.231588,2009-01-12,1458,1427.527697,...,0.055919,3.610772,3.552354,0.034750,"{""type"":""MultiPoint"",""coordinates"":[]}",137,1057,1,2009,1
3,"20090112_138,1057",317.700012,122.948921,137.568141,76.0,32.306502,22.015619,2009-01-12,1449,1421.492260,...,0.050070,3.635362,3.595614,0.024742,"{""type"":""MultiPoint"",""coordinates"":[]}",138,1057,1,2009,1
4,"20090112_137,1058",317.700012,19.856251,118.399804,102.0,75.860119,32.361242,2009-01-12,1434,1401.470238,...,0.137627,3.566329,3.494493,0.041712,"{""type"":""MultiPoint"",""coordinates"":[]}",137,1058,1,2009,1


In [7]:
#split the data into training and testing sets
train_df = final_df[final_df['year'] < 2021] #train on data before 2021
test_df = final_df[final_df['year'] >= 2021] #test on data from 2021 and after

#seperate them into inputs and outputs
col_drop = ['fire','system:index','date', '.geo', 'T21_max', 'T21_mean', 'T21_stdDev'] #these are the columns we can get rid of
X_train = train_df.drop(columns=col_drop)
x_test = test_df.drop(columns=col_drop)
y_train = train_df['fire']
y_test = test_df['fire']

In [8]:
RandForest = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42) 
RandForest.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [9]:
probs = RandForest.predict_proba(x_test)[:, 1] #probability of fire
threshold = 0.6 #we can change this threshold to be more or less sensitive to fire predictions
predictions = (probs >= threshold).astype(int) #convert probabilities to 0

print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

           0       0.92      1.00      0.96     41932
           1       0.45      0.01      0.02      3589

    accuracy                           0.92     45521
   macro avg       0.69      0.50      0.49     45521
weighted avg       0.88      0.92      0.88     45521



In [10]:
import joblib
joblib.dump(RandForest, 'fire_prediction_model.pkl')

['fire_prediction_model.pkl']